# 10. Transfer Learning Implementation with Local CIFAR-10

This notebook runs transfer learning on the CIFAR-10 files already present in this project at data/cifar-10-batches-py. It never downloads another dataset. TorchVision's pretrained ResNet-18 weights may be obtained if they are not cached; that is model initialization, not a dataset.

## Workflow

1. Load local CIFAR-10 with downloading disabled.
2. Inspect a pretrained ResNet-18.
3. Freeze the backbone, replace its classifier, and train the classifier.
4. Evaluate, unfreeze the final residual block, and fine-tune.
5. Compare with a small CNN trained from scratch on the same local training examples.

Defaults use deterministic subsets of the existing data for a practical CPU-friendly run. Increase sample counts and epochs in the configuration cell for a longer experiment. The official test set remains separate from training and validation.

In [1]:
from pathlib import Path
import random
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
SEED = 42
DATA_ROOT = Path("data")
CIFAR_DIR = DATA_ROOT / "cifar-10-batches-py"
TRAIN_SAMPLES = 5000
VALIDATION_SAMPLES = 1000
TEST_SAMPLES = 1000
BATCH_SIZE = 64
IMAGE_SIZE = 96
HEAD_EPOCHS = 2
FINETUNE_EPOCHS = 1
SCRATCH_EPOCHS = 2
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if not CIFAR_DIR.is_dir():
    raise FileNotFoundError(f"Local CIFAR-10 data not found: {CIFAR_DIR.resolve()}")

Device: cpu


## 1. Load and inspect the existing dataset

CIFAR-10 is already extracted into the project. Setting download=False makes a missing local dataset an explicit error rather than fetching another copy.

In [2]:
class_names = datasets.CIFAR10(root=str(DATA_ROOT), train=True, download=False).classes
num_classes = len(class_names)
print(f"Classes ({num_classes}): {class_names}")
train_count = min(TRAIN_SAMPLES, 50000 - VALIDATION_SAMPLES)
validation_count = min(VALIDATION_SAMPLES, 50000 - train_count)
permutation = torch.randperm(50000, generator=torch.Generator().manual_seed(SEED)).tolist()
train_indices = permutation[:train_count]
validation_indices = permutation[train_count:train_count + validation_count]
test_indices = torch.randperm(10000, generator=torch.Generator().manual_seed(SEED + 1))[:TEST_SAMPLES].tolist()
imagenet_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
scratch_transform = transforms.ToTensor()
transfer_train_data = datasets.CIFAR10(root=str(DATA_ROOT), train=True, download=False, transform=imagenet_transform)
transfer_test_data = datasets.CIFAR10(root=str(DATA_ROOT), train=False, download=False, transform=imagenet_transform)
train_loader = DataLoader(Subset(transfer_train_data, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
validation_loader = DataLoader(Subset(transfer_train_data, validation_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(Subset(transfer_test_data, test_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Local training subset: {len(train_indices)} | validation subset: {len(validation_indices)} | test subset: {len(test_indices)}")
print(f"CIFAR-10 classes: {num_classes}")

Classes (10): ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Local training subset: 5000 | validation subset: 1000 | test subset: 1000
CIFAR-10 classes: 10


## 2. Load and inspect pretrained ResNet-18

TorchVision provides an ImageNet-pretrained ResNet-18. CIFAR-10 is also a natural-image dataset, so its learned visual features are a reasonable starting point. We resize CIFAR images and apply the normalization expected by the pretrained weights.

In [3]:
weights = models.ResNet18_Weights.DEFAULT
model = models.resnet18(weights=weights)
print(model)
print(f"Original output classes: {model.fc.out_features}")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
model = model.to(device)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

## 3. Freeze the backbone and replace its classifier

All pretrained parameters are frozen, then the final fully connected layer is replaced with a randomly initialized 10-class head. Only this head is trainable in the first stage.

In [4]:
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, num_classes).to(device)
print(f"New head: {model.fc}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

New head: Linear(in_features=512, out_features=10, bias=True)
Trainable parameters: 5,130


## 4. Train the classifier head

The helper functions report epoch time, loss, and accuracy. Batch-normalization layers whose parameters are frozen are kept in evaluation mode so their running statistics do not drift during feature extraction.

In [5]:
criterion = nn.CrossEntropyLoss()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.inference_mode():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            total_loss += criterion(logits, labels).item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total
def train_epochs(model, loader, optimizer, epochs):
    history = []
    for epoch in range(epochs):
        model.train()
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d) and not any(p.requires_grad for p in module.parameters()):
                module.eval()
        total_loss = 0.0
        correct = 0
        total = 0
        start_time = time.perf_counter()
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
        epoch_seconds = time.perf_counter() - start_time
        val_loss, val_accuracy = evaluate(model, validation_loader)
        record = {"train_loss": total_loss / total, "train_accuracy": correct / total,
                  "validation_loss": val_loss, "validation_accuracy": val_accuracy,
                  "seconds": epoch_seconds}
        history.append(record)
        print(f"Epoch {epoch + 1}/{epochs} | train loss {record['train_loss']:.4f} | "
              f"train accuracy {record['train_accuracy']:.3f} | val accuracy {val_accuracy:.3f} | "
              f"{epoch_seconds:.1f}s")
    return history

In [6]:
head_optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
head_start = time.perf_counter()
head_history = train_epochs(model, train_loader, head_optimizer, HEAD_EPOCHS)
head_training_seconds = time.perf_counter() - head_start
head_test_loss, head_test_accuracy = evaluate(model, test_loader)
print(f"Frozen-backbone test accuracy: {head_test_accuracy:.3f} (loss {head_test_loss:.4f})")

Epoch 1/2 | train loss 1.1820 | train accuracy 0.602 | val accuracy 0.763 | 62.2s
Epoch 2/2 | train loss 0.6190 | train accuracy 0.791 | val accuracy 0.782 | 57.9s
Frozen-backbone test accuracy: 0.793 (loss 0.6184)


## 5. Unfreeze selected layers and fine-tune

Unfreeze ResNet's final residual block (layer4) and the classifier. Use a lower learning rate to adjust high-level features gradually while preserving earlier general-purpose filters.

In [7]:
for parameter in model.parameters():
    parameter.requires_grad = False
for name, parameter in model.named_parameters():
    if name.startswith("layer4.") or name.startswith("fc."):
        parameter.requires_grad = True
print(f"Trainable parameters after unfreezing: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
finetune_optimizer = torch.optim.Adam((p for p in model.parameters() if p.requires_grad), lr=1e-4)
finetune_start = time.perf_counter()
finetune_history = train_epochs(model, train_loader, finetune_optimizer, FINETUNE_EPOCHS)
finetune_training_seconds = time.perf_counter() - finetune_start
finetune_test_loss, finetune_test_accuracy = evaluate(model, test_loader)
print(f"Fine-tuned test accuracy: {finetune_test_accuracy:.3f} (loss {finetune_test_loss:.4f})")

Trainable parameters after unfreezing: 8,398,858
Epoch 1/1 | train loss 0.5693 | train accuracy 0.804 | val accuracy 0.841 | 88.9s
Fine-tuned test accuracy: 0.871 (loss 0.4083)


## 6. Train a small CNN from scratch on the same local data

This compact model starts with random weights and uses native 32x32 CIFAR images. It uses the same training and test indices as the transfer experiment. This is a practical comparison, not a perfectly compute-matched benchmark: the architectures and input sizes differ, and the subset/epoch settings are deliberately modest for a notebook run.

In [8]:
class ScratchCNN(nn.Module):
    def __init__(self, output_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, output_classes)
    def forward(self, images):
        return self.classifier(torch.flatten(self.features(images), 1))
transfer_train_data.transform = scratch_transform
transfer_test_data.transform = scratch_transform
scratch_train_loader = DataLoader(Subset(transfer_train_data, train_indices), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
scratch_test_loader = DataLoader(Subset(transfer_test_data, test_indices), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

scratch_model = ScratchCNN(num_classes).to(device)
scratch_optimizer = torch.optim.Adam(scratch_model.parameters(), lr=1e-3)
scratch_start = time.perf_counter()
for epoch in range(SCRATCH_EPOCHS):
    scratch_model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in scratch_train_loader:
        images, labels = images.to(device), labels.to(device)
        scratch_optimizer.zero_grad(set_to_none=True)
        logits = scratch_model(images)
        loss = criterion(logits, labels)
        loss.backward()
        scratch_optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)
    print(f"Scratch epoch {epoch + 1}/{SCRATCH_EPOCHS} | loss {total_loss / total:.4f} | accuracy {correct / total:.3f}")
scratch_training_seconds = time.perf_counter() - scratch_start
scratch_test_loss, scratch_test_accuracy = evaluate(scratch_model, scratch_test_loader)
print(f"Scratch CNN test accuracy: {scratch_test_accuracy:.3f} (loss {scratch_test_loss:.4f})")

Scratch epoch 1/2 | loss 1.8579 | accuracy 0.321
Scratch epoch 2/2 | loss 1.6003 | accuracy 0.427
Scratch CNN test accuracy: 0.319 (loss 1.8384)


## 7. Compare measured results

The following results come from selected subsets of local CIFAR-10. Run the cells to populate the table; no outcomes are assumed in advance.

In [9]:
import pandas as pd

comparison = pd.DataFrame([
    {"Approach": "ResNet-18 frozen backbone", "Test accuracy": head_test_accuracy, "Test loss": head_test_loss, "Training seconds": head_training_seconds},
    {"Approach": "ResNet-18 fine-tuned layer4", "Test accuracy": finetune_test_accuracy, "Test loss": finetune_test_loss, "Training seconds": finetune_training_seconds},
    {"Approach": "CNN trained from scratch", "Test accuracy": scratch_test_accuracy, "Test loss": scratch_test_loss, "Training seconds": scratch_training_seconds},
])
comparison

,Approach,Test accuracy,Test loss,Training seconds
0,ResNet-18 frozen backbone,0.793,0.618397,143.463624
1,ResNet-18 fine-tuned layer4,0.871,0.408346,100.891667
2,CNN trained from scratch,0.319,1.838404,20.006089


## 8. Training from scratch vs transfer learning

| Factor | Training from scratch | Transfer learning |
|---|---|---|
| Training time | Must learn useful visual features and the classifier from random initialization; commonly needs more training. | Starts with learned visual features; head-only training is usually quick, though the frozen backbone still performs forward computation. |
| Data requirement | Usually needs more labeled examples to learn robust features. | Often works well with fewer labeled examples because the backbone already encodes useful patterns. |
| Performance | Can be strong with enough relevant data and tuning; may underperform on a small dataset. | Often gives a better starting point and can improve after fine-tuning; domain mismatch can reduce the benefit. |
| Computational cost | All network parameters require gradient updates. | Feature extraction updates only the head; fine-tuning selected blocks uses less backpropagation than full training. |
| Generalization | Small datasets can lead to overfitting from random initialization. | Pretraining can provide a useful inductive bias, but does not guarantee better generalization. |

**Analysis prompts:** Which method reached higher test accuracy in your run? Did fine-tuning improve on the frozen backbone? Were the training-time differences worthwhile? Model size and input resolution differ, so timings are illustrative, not a controlled architecture benchmark.

## 9. Save the fine-tuned model for notebook 11

This saves model weights to an artifacts folder so notebook 11 can evaluate the result on the full local CIFAR-10 test set. It does not change or download the dataset.

In [10]:
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = ARTIFACT_DIR / "resnet18_cifar10_finetuned.pth"
torch.save(model.cpu().state_dict(), CHECKPOINT_PATH)
model.to(device)
print(f"Saved fine-tuned weights to {CHECKPOINT_PATH.resolve()}")

Saved fine-tuned weights to C:\Users\hemak\Sprint 10 - Advanced Deep Learning, CNNs, Transfer Learning, and Computer Vision\artifacts\resnet18_cifar10_finetuned.pth


## 10. Summary

You loaded existing local CIFAR-10 without downloading, used pretrained ResNet-18 as a feature extractor, trained its replacement classifier, fine-tuned layer4, and compared it with a small randomly initialized CNN. Metrics depend on the actual run, subset sizes, epochs, and hardware.